# Exercice 1 — Prédiction de la consommation électrique

Ce notebook implémente un Transformer qui utilise les 24 dernières heures pour prédire la puissance moyenne de l'heure suivante.

Le fichier `household_power_consumption.txt` doit être placé dans le même dossier que ce notebook. Il est disponible dans le jeu de données UCI *Individual Household Electric Power Consumption*.

La séparation est chronologique : 70 % pour l'entraînement, 15 % pour la validation et 15 % pour le test. La moyenne et l'écart-type sont calculés uniquement sur l'entraînement afin d'éviter une fuite d'information.

In [1]:
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## 1. Préparer les données

`Global_active_power` est une puissance moyenne exprimée en kW. Les mesures sont agrégées par heure. Une heure est conservée si elle possède au moins 45 mesures valides.

In [2]:
chemin = "household_power_consumption.txt"
try:
    df = pd.read_csv(
        chemin, sep=";", na_values=["?"], low_memory=False
    )
except FileNotFoundError as erreur:
    raise FileNotFoundError(
        "Placez household_power_consumption.txt dans le dossier du notebook."
    ) from erreur

df["datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M:%S",
)
puissance = pd.to_numeric(df["Global_active_power"], errors="coerce")
puissance.index = pd.DatetimeIndex(df["datetime"])
horaire = puissance.resample("h").agg(["mean", "count"])
serie = horaire["mean"].where(horaire["count"] >= 45).astype("float32")

n = len(serie)
fin_train, fin_val = int(0.70 * n), int(0.85 * n)
moyenne = serie.iloc[:fin_train].mean()
ecart_type = serie.iloc[:fin_train].std()
if not np.isfinite(moyenne) or not np.isfinite(ecart_type) or ecart_type == 0:
    raise ValueError("La série ne contient pas assez de mesures valides.")
valeurs = ((serie - moyenne) / ecart_type).to_numpy()

fenetre = 24
X, y, dates_cibles = [], [], []
for i in range(fenetre, n):
    sequence, cible = valeurs[i - fenetre:i], valeurs[i]
    if np.isfinite(sequence).all() and np.isfinite(cible):
        X.append(sequence)
        y.append(cible)
        dates_cibles.append(i)

X = torch.tensor(np.asarray(X), dtype=torch.float32).unsqueeze(-1)
y = torch.tensor(np.asarray(y), dtype=torch.float32).unsqueeze(-1)
dates_cibles = np.asarray(dates_cibles)
train_mask = dates_cibles < fin_train
val_mask = (dates_cibles >= fin_train) & (dates_cibles < fin_val)
test_mask = dates_cibles >= fin_val

def chargeur(mask, melanger=False):
    return DataLoader(
        TensorDataset(X[mask], y[mask]),
        batch_size=64,
        shuffle=melanger,
    )

train_loader = chargeur(train_mask, True)
val_loader = chargeur(val_mask)
test_loader = chargeur(test_mask)
print("Exemples :", len(X), "| train :", train_mask.sum(),
      "| validation :", val_mask.sum(), "| test :", test_mask.sum())
print("Forme d'une entrée :", tuple(X.shape[1:]))

FileNotFoundError: Placez household_power_consumption.txt dans le dossier du notebook.

## 2. Construire `EnergyTransformer`

Le modèle suit exactement l'architecture demandée :

`(batch, 24, 1) → Linear(1, 32) → positions → 2 encodeurs → Linear(32, 1)`.

La représentation de la dernière heure est utilisée pour la prédiction.

In [ ]:
class EnergyTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        dimension = 32
        self.projection = nn.Linear(1, dimension)
        self.positions = nn.Embedding(24, dimension)
        couche = nn.TransformerEncoderLayer(
            d_model=dimension,
            nhead=4,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True,
        )
        self.encodeur = nn.TransformerEncoder(couche, num_layers=2)
        self.sortie = nn.Linear(dimension, 1)

    def forward(self, x):
        longueur = x.shape[1]
        positions = torch.arange(longueur, device=x.device)
        z = self.projection(x) + self.positions(positions)[None, :, :]
        z = self.encodeur(z)
        return self.sortie(z[:, -1, :])


modele = EnergyTransformer().to(device)
critere = nn.MSELoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.001)
print(modele)
print("Sortie :", tuple(modele(X[:2].to(device)).shape))

## 3. Entraîner et conserver le meilleur modèle

La perte de validation sert uniquement à choisir le meilleur état. Les données de validation ne servent pas à mettre à jour les paramètres.

In [ ]:
def perte_moyenne(loader):
    modele.eval()
    total, nombre = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            perte = critere(modele(xb.to(device)), yb.to(device))
            total += perte.item() * len(xb)
            nombre += len(xb)
    return total / nombre


meilleure_val = float("inf")
meilleur_etat = None
historique = []
for epoch in range(1, 21):
    modele.train()
    for xb, yb in train_loader:
        perte = critere(modele(xb.to(device)), yb.to(device))
        optimiseur.zero_grad()
        perte.backward()
        optimiseur.step()
    val_loss = perte_moyenne(val_loader)
    historique.append(val_loss)
    if val_loss < meilleure_val:
        meilleure_val = val_loss
        meilleur_etat = deepcopy(modele.state_dict())
    print(f"Époque {epoch:02d} | validation MSE : {val_loss:.5f}")

if meilleur_etat is None:
    raise RuntimeError("Aucun modèle valide n'a été obtenu.")
modele.load_state_dict(meilleur_etat)

## 4. Évaluer en kW et comparer les références

Les prédictions sont reconverties avec `valeur * ecart_type + moyenne`. Les deux références sont :

- **persistance** : dernière observation disponible (`X[:, -1]`) ;
- **référence journalière** : observation 24 heures auparavant (`X[:, 0]`).

In [ ]:
def predictions_test():
    modele.eval()
    pred, vrai, entrees = [], [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            pred.append(modele(xb.to(device)).cpu())
            vrai.append(yb)
            entrees.append(xb)
    return torch.cat(pred).squeeze(1).numpy(), torch.cat(vrai).squeeze(1).numpy(), torch.cat(entrees).squeeze(1).numpy()


pred_norm, vrai_norm, entrees_norm = predictions_test()
pred = pred_norm * ecart_type + moyenne
vrai = vrai_norm * ecart_type + moyenne
persistance = entrees_norm[:, -1] * ecart_type + moyenne
journaliere = entrees_norm[:, 0] * ecart_type + moyenne

def metriques(reference, cible):
    erreur = reference - cible
    return float(np.mean(np.abs(erreur))), float(np.sqrt(np.mean(erreur ** 2)))

resultats = pd.DataFrame(
    [metriques(pred, vrai), metriques(persistance, vrai), metriques(journaliere, vrai)],
    index=["Transformer", "Persistance", "Référence journalière"],
    columns=["MAE (kW)", "RMSE (kW)"],
)
resultats

In [ ]:
# Tracer au maximum sept jours consécutifs du test.
nombre_heures = min(7 * 24, len(vrai))
plt.figure(figsize=(12, 4))
plt.plot(vrai[:nombre_heures], label="Observations", linewidth=2)
plt.plot(pred[:nombre_heures], label="Transformer", linestyle="--")
plt.plot(persistance[:nombre_heures], label="Persistance", alpha=0.7)
plt.title("Consommation électrique : sept jours du test")
plt.xlabel("Heure de test")
plt.ylabel("Puissance moyenne (kW)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Questions de réflexion — réponses

1. **Le Transformer améliore-t-il les références ?**  
   Comparez les lignes du tableau : il améliore une référence si sa MAE et sa RMSE sont plus faibles. Le résultat dépend de la période du test et des données manquantes.

2. **Une fenêtre de 168 heures est-elle utile ?**  
   Elle peut capturer les habitudes hebdomadaires, mais augmente le nombre de paramètres et la difficulté d'apprentissage. Il faut modifier les positions, entraîner à nouveau et comparer sur le même test.

3. **Pourquoi ajouter l'heure et le jour de la semaine ?**  
   Ce sont des variables calendaires qui rendent explicites les cycles journaliers et hebdomadaires, difficiles à déduire avec seulement 24 valeurs.

4. **Pourquoi normaliser uniquement sur l'entraînement ?**  
   Utiliser la validation ou le test pour calculer la moyenne introduirait une information future dans l'entraînement et donnerait une évaluation trop optimiste.

5. **Comment prédire une énergie en kWh ?**  
   Pour une moyenne de puissance sur une heure, l'énergie correspond à `puissance_kW × 1 h`. Pour des intervalles différents, il faut multiplier chaque mesure par sa durée en heures et agréger les énergies, plutôt que moyenner directement les puissances.